# 01 — Exploratory Data Analysis (MovieLens 25M)

This notebook explores the raw MovieLens 25M dataset: rating distributions, sparsity, long-tail of items, genre coverage, and temporal activity. These findings motivate the design decisions in `src/data/loader.py` (time-based split) and the model choices.

**Prerequisite:** `make download-data` (or `python -m src.data.download`)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

from src.data.loader import load_ratings, load_movies, load_tags

ratings = load_ratings()
movies = load_movies()
tags = load_tags()

print(f'Ratings: {len(ratings):,}')
print(f'Movies:  {len(movies):,}')
print(f'Tags:    {len(tags):,}')
ratings.head()

## Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=ratings, x='rating', ax=ax, color='steelblue')
ax.set_title('Rating value distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

print(f'Mean rating: {ratings.rating.mean():.3f}')
print(f'Median:      {ratings.rating.median():.3f}')
print(f'Std:         {ratings.rating.std():.3f}')
print(f'Fraction >= 4.0 (implicit positive): {(ratings.rating >= 4.0).mean():.1%}')

## Sparsity

Recommender systems are defined by sparsity. We compute the density of the user-item matrix.

In [ ]:
n_users = ratings.userId.nunique()
n_items = ratings.movieId.nunique()
n_possible = n_users * n_items
n_observed = len(ratings)
sparsity = 1 - n_observed / n_possible

print(f'Users:    {n_users:,}')
print(f'Items:    {n_items:,}')
print(f'Possible: {n_possible:,}')
print(f'Observed: {n_observed:,}')
print(f'Sparsity: {sparsity:.6f}  (density = {1 - sparsity:.6f})')

## Long tail of item popularity

Most items get very few ratings. This drives the cold-start problem that the content-based model addresses.

In [ ]:
item_counts = ratings.movieId.value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(item_counts, bins=100, log=True, color='steelblue')
axes[0].set_title('Item rating count distribution (log y)')
axes[0].set_xlabel('# ratings per item')
axes[0].set_ylabel('# items (log)')

ranked = np.sort(item_counts.values)[::-1]
axes[1].loglog(np.arange(1, len(ranked) + 1), ranked, color='steelblue')
axes[1].set_title('Long-tail curve (log-log)')
axes[1].set_xlabel('Item rank')
axes[1].set_ylabel('# ratings')

plt.tight_layout(); plt.show()

print(f'Top 1% items cover {item_counts.head(int(n_items * 0.01)).sum() / n_observed:.1%} of all ratings')
print(f'Bottom 50% items cover {item_counts.tail(int(n_items * 0.5)).sum() / n_observed:.1%} of all ratings')

## User activity distribution

In [ ]:
user_counts = ratings.userId.value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(user_counts, bins=100, log=True, color='seagreen')
ax.set_title('User activity distribution (log y)')
ax.set_xlabel('# ratings per user')
ax.set_ylabel('# users (log)')
plt.tight_layout(); plt.show()

print(f'Median ratings/user: {user_counts.median():.0f}')
print(f'Max ratings/user:    {user_counts.max()}')
print(f'Min ratings/user:    {user_counts.min()}')

## Genre coverage

In [ ]:
genre_split = movies.genres.str.split('|')
all_genres = [g for parts in genre_split for g in parts]
genre_counts = pd.Series(all_genres).value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
genre_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Genre frequency (by movie count)')
ax.set_ylabel('# movies')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

## Temporal activity

Ratings span a long time window. This is exactly why we use a **time-based split** rather than random K-fold: a random split leaks future interactions into training and inflates ranking metrics.

In [ ]:
ratings['dt'] = pd.to_datetime(ratings.timestamp, unit='s')
per_year = ratings.dt.dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
per_year.plot(kind='bar', ax=ax, color='slateblue')
ax.set_title('Ratings per year')
ax.set_ylabel('# ratings')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

print(f'Earliest: {ratings.dt.min()}')
print(f'Latest:   {ratings.dt.max()}')

## Key takeaways

1. **Sparsity ~99.998%** — extreme, typical for recommenders. Matrix factorization and NCF are designed for this regime.
2. **Strong long tail** — the bottom 50% of items get almost no ratings. Content-based features (TF-IDF on tags/genres) are needed for cold-start items.
3. **Rating skew toward 4-5** — supports the implicit-feedback formulation (binarize at 4.0) used by NCF.
4. **Wide temporal range** — justifies the time-based train/test split implemented in `src/data/loader.py`.

Next: `02_baselines.ipynb` — train SVD + popularity baseline and inspect results.